# Excitation Index, Recomputed (R2 revision)

Recomputes the four-component Excitation Index (EI) reported in Table 4 and Table S10 on the corrected engagement file (view counts at raw scale, see the README Revision Notes). Two changes from the first computation:

1. Tweets with no recorded engagement metrics (all five fields empty) are excluded. They were previously counted as zero, which pulled the post-shift engagement median down, most strongly for Parliament & Protest.
2. The post-shift engagement component uses the same post-shift window as the other analyses (28 August to 29 September 2025, with 27 August excluded).

Components per narrative (Section 3.6):

- **mean diffusion score**: each of the five engagement metrics is log1p-transformed and z-scored across tweets, the tweet score is the mean of the five, and the narrative value is the mean over its tweets;
- **maximum and mean community hop distance**: unchanged from the first computation (`excitation_integrated_table.csv`, Table S9);
- **post-shift median engagement per tweet**: median of the five metrics summed per tweet, over post-shift tweets.

Each component is standardized across the seven narratives (population SD) and the EI is their unweighted mean. The robustness check drops the post-shift engagement component.

Inputs: `Data_with_community_hashed.csv` (derived-data/hashed/), `excitation_integrated_table.csv` and `excitation_index.csv` (derived-data/).

In [1]:
import numpy as np, pandas as pd
df = pd.read_csv('Data_with_community_hashed.csv', sep=';', encoding='utf-8-sig')   # derived-data/hashed/
M = ['view_count', 'reply_count', 'retweet_count', 'quote_count', 'favorite_count']
df['day'] = pd.to_datetime(df['date'], format='%d/%m/%Y %H:%M').dt.normalize()
lab = df['narrative'].notna()
missing = df[M].isna().all(axis=1)
print(f'rows: {len(df)} | narrative-labelled: {lab.sum()} | labelled without engagement: {(lab & missing).sum()}')
print('date range:', df['day'].min().date(), 'to', df['day'].max().date())
print('\nmedian views per tweet (Table S1 Panel A):')
print(df[lab].groupby('narrative')['view_count'].median().sort_values(ascending=False).to_string())

rows: 14248 | narrative-labelled: 10777 | labelled without engagement: 2267
date range: 2025-08-01 to 2025-09-29

median views per tweet (Table S1 Panel A):
narrative
Affan Kurniawan       375.0
Keamanan & Respons    264.0
Politik & Tuntutan    148.5
Demo & DPR            136.0
Gerakan/Hashtag        98.0
Kekerasan Aparat       90.0
Ekonomi Rakyat         82.0


In [2]:
# tweet-level inputs, on narrative-labelled tweets with recorded engagement
d = df[lab & ~missing].copy()
L = np.log1p(d[M])
d['diffusion_score'] = ((L - L.mean()) / L.std()).mean(axis=1)
d['engagement_sum'] = d[M].sum(axis=1)
post = (d['day'] >= '2025-08-28') & (d['day'] <= '2025-09-29')
print(f'tweets used: {len(d)} | post-shift tweets: {post.sum()}')

tweets used: 8510 | post-shift tweets: 4566


In [3]:
# components, EI, and the robustness check without the engagement component (Table 4, Table S10)
label = {'Affan Kurniawan': 'Affan Kurniawan', 'Keamanan & Respons': 'Security & Response',
         'Demo & DPR': 'Parliament & Protest', 'Kekerasan Aparat': 'State Apparatus Violence',
         'Politik & Tuntutan': 'Politics & Demands', 'Ekonomi Rakyat': 'Economic Grievance',
         'Gerakan/Hashtag': 'Movement Coordination'}
hops = pd.read_csv('excitation_integrated_table.csv').set_index('narrative')[['max_hop', 'avg_hop']]   # derived-data/
raw = pd.DataFrame({
    'mean_diffusion': d.groupby('narrative')['diffusion_score'].mean(),
    'max_hop': hops['max_hop'],
    'avg_hop': hops['avg_hop'],
    'post_median_engagement': d[post].groupby('narrative')['engagement_sum'].median(),
})
z = lambda s: (s - s.mean()) / s.std(ddof=0)
comp = list(raw.columns)
raw['EI'] = sum(z(raw[c]) for c in comp) / 4
raw['EI_without_engagement'] = sum(z(raw[c]) for c in comp[:3]) / 3
raw = raw.sort_values('EI', ascending=False)
raw['rank'] = range(1, len(raw) + 1)
raw['rank_without_engagement'] = raw['EI_without_engagement'].rank(ascending=False).astype(int)
raw.insert(0, 'label', raw.index.map(label))
print(raw.round(3).to_string())
raw.round(4).to_csv('excitation_index_recomputed.csv')

                                       label  mean_diffusion  max_hop  avg_hop  post_median_engagement     EI  EI_without_engagement  rank  rank_without_engagement
narrative                                                                                                                                                          
Affan Kurniawan              Affan Kurniawan           0.438        7     2.84                   391.0  1.933                  1.954     1                        1
Keamanan & Respons       Security & Response           0.170        5     2.63                   304.0  0.822                  0.745     2                        2
Demo & DPR              Parliament & Protest          -0.033        5     2.27                   196.0  0.038                  0.040     3                        3
Kekerasan Aparat    State Apparatus Violence          -0.088        4     1.93                   108.5 -0.668                 -0.627     4                        5
Politik & Tuntut

In [4]:
# comparison with the first computation (excitation_index.csv)
old = pd.read_csv('excitation_index.csv').set_index('narrative')   # derived-data/
cmp = pd.DataFrame({
    'EI_first': old['excitation_index'], 'rank_first': old['rank'],
    'EI_recomputed': raw['EI'], 'rank_recomputed': raw['rank'],
    'post_engagement_first': old['post_median_eng_per_tweet'],
    'post_engagement_recomputed': raw['post_median_engagement'],
}).loc[raw.index]
print(cmp.round(2).to_string())

                    EI_first  rank_first  EI_recomputed  rank_recomputed  post_engagement_first  post_engagement_recomputed
narrative                                                                                                                  
Affan Kurniawan         1.97           1           1.93                1                  360.0                       391.0
Keamanan & Respons      0.79           2           0.82                2                  227.5                       304.0
Demo & DPR             -0.23           3           0.04                3                    0.0                       196.0
Kekerasan Aparat       -0.57           5          -0.67                4                   68.0                       108.5
Politik & Tuntutan     -0.71           7          -0.68                5                   23.0                       110.0
Ekonomi Rakyat         -0.56           4          -0.69                6                   56.0                        86.5
Gerakan/